# 07 - Discovered indicators

The indicators the network learned, drawn on the price of one window next to the same indicators at their
textbook periods: the discovered indicators are what a run delivers (VISION), and the prediction and trading
quality in 01-04 are the evidence that they are good. Each copy of every registered family is drawn twice on the
same window: solid at the period the served model applied to that window, dashed at its configured textbook
period. The families are moving average, MACD, RSI, Bollinger, ATR, stochastic, Williams %R, Keltner, OBV,
VWAP, MFI, ADX, CCI and Donchian. The original four use the numpy formulas on the raw close; the other ten are
drawn from each family's outputs on the model window. Lines that smooth use an EWMA with alpha = 2 / (period + 1),
started at the window's first bar.

Set `WINDOW` to a window index, or to "typical" (closest to the block's median periods), "longest" / "shortest"
(where the per-window adjustment stretches or shrinks the periods most) or "last"; `BLOCK` to another block of
the run's fold. Nothing here writes to the run.

In [ ]:
# Parameters
RUN_DIR = None                  # a runs/<id> directory; None -> the newest run under RUNS_DIR
RUNS_DIR = "../runs"
CSV_PATH = "../binance_btcusdt_1min_ccxt.csv"
BLOCK = "test"                  # "train", "val", "cal" or "test"
WINDOW = "typical"              # a window index, or "typical" / "longest" / "shortest" / "last"

In [ ]:
from neural_trade.data.processor import split_arrays
from neural_trade.experiments.dataset import verify_run_dataset
from neural_trade.notebook import pick_run
from neural_trade.registries.visualizations import Visualizations
from neural_trade.serving.predictor import Predictor
from neural_trade.visualization import analytics_tables as AT
from neural_trade.visualization.discovered_indicators import discovered_table
from neural_trade.visualization.indicator_evolution import applied_periods

run_dir = pick_run(RUN_DIR, RUNS_DIR)   # newest run with a serving bundle, or a clear error
print("run:", run_dir)
predictor = Predictor.from_artifacts(run_dir / "artifacts")
cfg = predictor.config.copy(CSV_PATH=CSV_PATH)
verify_run_dataset(run_dir, cfg)          # the CSV above must be the data the run used (NT-041)
blocks = split_arrays(cfg)
block = blocks[BLOCK]
times = blocks["df"]["timestamp"].to_numpy()[block["anchor_bar"]]
metrics = run_dir / "metrics.jsonl"
applied = applied_periods(predictor, block["X_model"], block=BLOCK)   # the period each window gets (served weights)
print(f"{len(block['X']):,} {BLOCK} windows of {cfg.LOOKBACK} bars; served weights: epoch",
      predictor.bundle.meta.get("weights_epoch"))

## The learned indicators on price

Top: the block's close, one point per window, with the drawn window shaded. The grid: one row per family, one
column per copy; each panel's heading gives the learned and the textbook period (MACD: fast / slow / signal).
Below it, each base period over training (from the recorded start, the dashed line is its textbook value) and,
right of the last epoch, how the period the model applies varies across the block's windows: 5-95% and the middle
50% of the windows, their median, this window's period and the served base period. A period longer than the
window has not warmed up by the window's end (the EWMA starts at the window's first bar); the table's
"applied > lookback" column counts those windows.

In [ ]:
Visualizations.build("discovered_indicators", block["X"], cfg, applied=applied, metrics=metrics, window=WINDOW,
                     times=times, ohlcv=block["X_model"]).show()

## Learned against textbook periods

The table behind the figure, in bars: the served base period (the trained logit alone) and the median applied
period, each with its change against the textbook period; the applied range over the block's windows; this
window's period; the range the base period took during training; the share of windows whose applied period is
longer than the window; and whether the base period sits at a clip bound.

In [ ]:
AT.styled(discovered_table(applied, cfg, metrics=metrics, window=WINDOW), digits=2)

## Every learned period over training

One panel per family: each learned period (a MACD's fast, slow and signal, a Keltner's period and ATR period, a
stochastic's K and D) against the epochs, from its textbook start (dashed). Families whose periods are not in the
run's log are named in the subtitle. This is the second figure of the run's `indicator_report.html`.

In [ ]:
Visualizations.build("indicator_family_periods", metrics, cfg, applied=applied).show()

## Which indicators the model uses: grouped permutation importance

For each family instance, the channels of that instance are shuffled across the **validation** block's windows and
the model is scored again: the bar is how much the per-window loss rises, and one panel per horizon gives how much
the direction AUC falls (the hover also gives the hit-rate drop). Whiskers are the 2.5 and 97.5 percentiles of a
moving-block bootstrap (block of at least the longest horizon) that recomputes the loss and the AUC on every
resample (D-012); a bar whose whiskers cross zero is not distinguishable from an unused indicator. This is a
read-out after training: nothing in training reads it. It is the third figure of `indicator_report.html`.

In [ ]:
from neural_trade.serving.indicator_report import indicator_importance

Visualizations.build("permutation_importance", indicator_importance(predictor, blocks["val"]), cfg).show()